**TUGAS 5 PRAKTIKUM BIGDATA (ERNY KURNIAWATI/2505060004)**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/23 18:42:42 WARN Utils: Your hostname, ernykurniawati resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/23 18:42:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 18:42:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [7]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


**MENYIAPKAN DATASET**

In [8]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col ("harga_satuan"))
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target_cabang = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.show(5)
df_target_cabang.show()


+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      400

**A. JOIN DAN PERBANDINGAN TARGET**

In [9]:
from pyspark.sql.functions import sum as apark_sum
df_total = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
persentase_pencapaian = df_total.join(df_target_cabang, on="kota", how="inner")
persentase_pencapaian = persentase_pencapaian.withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100)
persentase_pencapaian = persentase_pencapaian.orderBy(col("pencapaian_persen").desc())
persentase_pencapaian.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. WINDOW FUNCTIONS**

In [10]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc
df_kategori = df_transaksi.groupby("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
window_kategori = Window.partitionBy("kota").orderBy(desc("total_pendapatan"))
peringkat_pendapatan = df_kategori.withColumn("peringkat", row_number().over(window_kategori))
peringkat_pendapatan = peringkat_pendapatan.filter(col("peringkat") == 1)
peringkat_pendapatan.show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



**C. SPARK SQL**

In [11]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target_cabang.createOrReplaceTempView("target")

data_ringkasan = spark.sql("""
    SELECT
        t.kota,
        tg.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    INNER JOIN target tg
        ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

data_ringkasan.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D.KESIMPULAN MINIMAL 100 KATA** 

Berdasarkan hasil analisis, cabang Purworejo memiliki pencapaian target paling tinggi, yaitu 152,17%. Total pendapatannya sebesar Rp45.650.000 dari target Rp30.000.000. Purworejo juga memiliki 116 transaksi, dengan kategori pendapatan tertinggi yaitu Kesehatan & Kecantikan sebesar Rp10.075.000. Sedangkan Semarang memiliki pencapaian target paling rendah, yaitu 69,41%, dengan total pendapatan Rp38.175.000 dari target Rp55.000.000. Jumlah transaksi di Semarang sebanyak 93 transaksi dan kategori dengan pendapatan tertinggi adalah Rumah Tangga sebesar Rp11.125.000. Dari hasil tersebut, dapat diketahui bahwa Purworejo sudah mencapai target dengan baik, sedangkan Semarang masih perlu meningkatkan pendapatannya agar dapat mencapai target.